# Stg Layer — search_results (Incremental)

Read **search_results** từ Bronze (`prod.bronze.search_results_raw`, append-only copy of Lakebase Postgres) (append-only discovery log — 1 row / (video, keyword, search run), không update) → MERGE 1:1 vào Delta staging table. Dùng MERGE trên `id` (thay vì INSERT thuần) để re-run sau lỗi vẫn idempotent, dù nguồn không bao giờ update rows.

**Incremental Flow:**
1. Lấy watermark (`MAX(fetched_at)`) từ Delta table
2. SELECT từ `prod.bronze.search_results_raw` (Bronze): chỉ rows có `fetched_at > watermark`
3. MERGE vào `prod.silver.stg_search_results` (UPSERT on `id`)
4. Verify row count

**Kiến trúc:** Databricks App → user search/save YouTube data → INSERT/UPDATE vào Postgres → `Analytic/bronze/ingest_raw` (append vào Bronze) → Job notebook (this) → SELECT Bronze (incremental) → MERGE vào Delta (stg layer, 1:1 với nguồn).


> **Source change (sub-task 3):** reads `prod.bronze.search_results_raw` instead of `lakebase_pg` and writes `prod.silver.stg_search_results`. Bronze keeps every read, so rows are deduped to the latest read per grain key before the MERGE.

In [0]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

SOURCE_TABLE = f"{BRONZE}.search_results_raw"  # Bronze: append-only copy of every Postgres read
DELTA_TABLE = f"{SILVER}.stg_search_results"
MERGE_KEY = "id"

print(f"Bronze source: {SOURCE_TABLE}")
print(f"Delta target: {DELTA_TABLE}")

In [0]:
# --- Get watermark from Delta table (last successful ETL) ---
try:
    watermark_row = spark.sql(f"SELECT MAX(fetched_at) AS wm FROM {DELTA_TABLE}").collect()[0]
    watermark = watermark_row["wm"] if watermark_row else None
except Exception:
    watermark = None

if watermark:
    # Incremental: only rows changed after watermark
    # Use Spark SQL — timestamps stay in Spark's timezone (UTC), no JDBC mismatch
    src_df = spark.sql(f"""
        SELECT * FROM {SOURCE_TABLE}
        WHERE fetched_at > '{watermark.strftime("%Y-%m-%d %H:%M:%S.%f")}'
    """)
    print(f"💧 Watermark: {watermark}")
    print("Incremental load: rows after watermark")
else:
    # Full load (first run or Delta empty)
    src_df = spark.sql(f"SELECT * FROM {SOURCE_TABLE}")
    print("📦 Full load (no watermark — first run)")

src_count = src_df.count()
print(f"Read {src_count} rows from Bronze ✅")
if src_count > 0:
    display(src_df.limit(10))

In [0]:
# --- Add etl_date and op_type to source data ---
from pyspark.sql.functions import current_timestamp, lit, col
# --- Keep the latest Bronze read per grain key (Bronze holds every read) ---
from pyspark.sql import Window
from pyspark.sql.functions import row_number
latest_read = Window.partitionBy("id").orderBy(col("ingest_ts").desc())
src_df = (
    src_df.withColumn("_rn", row_number().over(latest_read))
    .filter("_rn = 1")
    .drop("_rn", "ingest_ts", "ingest_date")
)
src_df = src_df.withColumn("etl_date", current_timestamp())
src_df = src_df.withColumn("op_type", lit("INSERT"))

# --- Create Delta table if not exists (1:1 with source + ETL metadata) ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DELTA_TABLE} (
        id             BIGINT,
        video_id       STRING,
        channel_id     STRING,
        title          STRING,
        published_at   TIMESTAMP,
        search_keyword STRING,
        fetched_at     TIMESTAMP,
        search_order   STRING,
        result_rank    INT,
        etl_date       TIMESTAMP,
        op_type        STRING
    )
    USING DELTA
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print(f"Delta table {DELTA_TABLE} ready ✅")

# --- Migrate: search_order / result_rank added by the app on 2026-10-26 (older rows stay NULL) ---
existing_cols = [f.name for f in spark.table(DELTA_TABLE).schema.fields]
if "search_order" not in existing_cols:
    spark.sql(f"ALTER TABLE {DELTA_TABLE} ADD COLUMNS (search_order STRING, result_rank INT)")
    print(f"Added search_order / result_rank to {DELTA_TABLE} ✅")
for c, t in [("search_order", "string"), ("result_rank", "int")]:
    if c not in src_df.columns:  # Bronze rows read before the app saved them
        src_df = src_df.withColumn(c, lit(None).cast(t))

# --- MERGE via SQL (compact, op_type tracking) ---
src_df.createOrReplaceTempView("src_source")

spark.sql(f"""
    MERGE INTO {DELTA_TABLE} AS t
    USING src_source AS s ON t.id = s.id
    WHEN MATCHED THEN UPDATE SET
        video_id=s.video_id,
        channel_id=s.channel_id,
        title=s.title,
        published_at=s.published_at,
        search_keyword=s.search_keyword,
        fetched_at=s.fetched_at,
        search_order=s.search_order,
        result_rank=s.result_rank,
        etl_date=s.etl_date, op_type='UPDATE'
    WHEN NOT MATCHED THEN INSERT *
""")

merged_count = spark.sql(f"SELECT COUNT(*) FROM {DELTA_TABLE}").collect()[0][0]
print(f"✅ Merged into {DELTA_TABLE}: {merged_count} total rows")

In [0]:
# --- Verify ---
total_delta = spark.sql(f"SELECT COUNT(*) as cnt FROM {DELTA_TABLE}").collect()[0]["cnt"]
last_etl = spark.sql(f"SELECT MAX(etl_date) as etl FROM {DELTA_TABLE}").collect()[0]["etl"]

op_counts = spark.sql(f"SELECT op_type, COUNT(*) as cnt FROM {DELTA_TABLE} GROUP BY op_type").collect()
op_breakdown = ", ".join(f"{r['op_type']}: {r['cnt']}" for r in op_counts)

print(f"📊 Total rows in {DELTA_TABLE}: {total_delta}")
print(f"📊 Last ETL date: {last_etl}")
print(f"📊 Op type breakdown: {op_breakdown}")
print(f"📊 Rows loaded this run: {src_count}")
print("\n--- stg_search_results ETL complete ---")